# Atelier Python Session 7 : la visualisation de données

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stage-to-data/hackspace-shs/blob/main/cours-python-resources/session-7/Atelier%20Python%20Session%207%20-%20Data%20vis.ipynb)

Un bon graphique vaut souvent mieux qu'un long tableau de chiffres ! Aujourd'hui, nous allons apprendre à visualiser
des données avec [matplotlib](https://matplotlib.org/), le package de référence pour faire des graphiques en Python,
puis à manipuler des tableaux de données avec [pandas](https://pandas.pydata.org/).

Il existe un deuxième notebook pour cette séance, **Atelier Python Session 7 - Data vis 2 - scikit-learn**, qui
présente la réduction de dimension et le clustering, pour visualiser des données plus complexes.

## Préparation

Exécutez cette cellule : elle crée les deux fichiers CSV utilisés dans ce notebook (`basic-data.csv` et
`other_data.csv`). Pas besoin de comprendre comment `other_data.csv` est fabriqué (il utilise `numpy`, que nous verrons
dans le deuxième notebook) : ce sont des points répartis en trois spirales.

In [ ]:
import os
import csv
import numpy as np

with open(os.path.join(os.getcwd(), "basic-data.csv"), "w", encoding="utf-8") as f:
  f.write("1, 2, 3, 4, 5, 6\n40, 30, 10, 5, 5, 10\n1, 3, 9, 10, 6, 8\n")

generateur = np.random.default_rng(0)
with open(os.path.join(os.getcwd(), "other_data.csv"), "w", encoding="utf-8", newline="") as f:
  writer = csv.writer(f)
  for classe in [1, 2, 3]:
    t = np.linspace(0.3, 3 * np.pi, 200)
    angle = t + (classe - 1) * 2 * np.pi / 3
    x = t * np.cos(angle) + generateur.normal(0, 0.3, 200)
    y = t * np.sin(angle) + generateur.normal(0, 0.3, 200)
    for i in range(200):
      writer.writerow([round(x[i], 4), round(y[i], 4), classe])

print("Fichiers créés !")

## 1. Ouvrir les données

Commençons par lire `basic-data.csv` avec le package `csv`, comme en séance 3. Ce fichier contient trois lignes de six
nombres.

In [ ]:
chemin = os.path.join(os.getcwd(), "basic-data.csv")

donnees = []
with open(chemin, encoding="utf-8") as f:
  csv_reader = csv.reader(f)
  for row in csv_reader:
    donnees.append(row)

print(donnees)

## 2. Premiers graphiques avec matplotlib

On importe le module `pyplot` de matplotlib, qui contient les fonctions pour dessiner. Par convention, on lui donne un
nom plus court avec le mot clé `as` : `plt`. On écrira donc `plt.plot()` au lieu de `matplotlib.pyplot.plot()`.

La fonction `plt.plot(x, y)` trace une courbe : on lui donne la liste des abscisses (l'axe horizontal) et la liste des
ordonnées (l'axe vertical). On peut ensuite ajouter un titre et le nom des axes, puis afficher le graphique avec
`plt.show()`.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(donnees[0], donnees[1])
plt.title("Mon super graphique")
plt.xlabel("Axe x")
plt.ylabel("Axe y")

plt.show()

Regardez bien l'axe vertical : les valeurs sont dans le désordre (`40`, `30`, `10`, `5` en partant du bas) ! C'est
parce que `csv.reader` lit toutes les cellules comme des **strings** (`" 30"`, et non `30`). matplotlib ne sait pas que
ce sont des nombres : il les traite comme des étiquettes, qu'il place dans l'ordre où il les rencontre.

La solution : **convertir les données en nombres** avant de les dessiner. (Vous pouvez vous entraîner à le faire vous-même
dans l'exercice 1 du notebook d'exercices.)

In [ ]:
nombres = []
for ligne in donnees:
  ligne_convertie = []
  for valeur in ligne:
    ligne_convertie.append(int(valeur))  # int(" 30") vaut 30 : les espaces sont ignorés
  nombres.append(ligne_convertie)

x = nombres[0]
y = nombres[1]
z = nombres[2]
print(x, y, z)

plt.plot(x, y)
plt.title("Mon super graphique, réparé")
plt.xlabel("Axe x")
plt.ylabel("Axe y")
plt.show()

### D'autres types de graphiques

matplotlib propose de nombreux types de graphiques, qui s'utilisent tous de la même manière :
- `plt.scatter(x, y)` : un **nuage de points** ;
- `plt.bar(x, hauteurs)` : un **diagramme en barres** ;
- `plt.pie(valeurs, labels=...)` : un **diagramme circulaire** (camembert) ;
- `plt.hist(valeurs)` : un **histogramme** (la répartition des valeurs)...

On peut aussi superposer plusieurs tracés sur un même graphique, en les appelant l'un après l'autre avant
`plt.show()`, et ajouter une légende avec `label=` et `plt.legend()`. De nombreux arguments permettent de personnaliser
l'apparence : `color`, `marker`, `linestyle`...

La [galerie de matplotlib](https://matplotlib.org/stable/gallery/index.html) montre tout ce qu'il est possible de faire,
avec le code de chaque exemple.

In [ ]:
plt.scatter(x, y)
plt.title("Nuage de points")
plt.show()

In [ ]:
plt.bar(x, y)
plt.title("Diagramme en barres")
plt.show()

In [ ]:
# Deux courbes sur le même graphique, avec une légende :
plt.plot(x, y, label="deuxième ligne", color="tab:blue", marker="o")
plt.plot(x, z, label="troisième ligne", color="tab:orange", linestyle="--")
plt.legend()
plt.title("Deux courbes")
plt.show()

### Enregistrer un graphique

Pour enregistrer un graphique dans un fichier image, on utilise `plt.savefig()`, avec le nom du fichier. Le format
dépend de l'extension : `.png`, `.jpg`, `.pdf`, `.svg`...

⚠ Il faut appeler `plt.savefig()` **avant** `plt.show()` : après l'affichage, la figure est vidée, et on enregistrerait
une image blanche !

In [ ]:
plt.pie(y, labels=x)
plt.title("Diagramme circulaire")
plt.savefig("camembert.png")
plt.show()

Ouvrez le panneau des fichiers de Colab : l'image `camembert.png` est là.

### Plusieurs graphiques dans une figure : `subplots`

Jusqu'ici, nous avons utilisé les fonctions de `plt` directement. Pour des figures plus complexes, on utilise plutôt
`plt.subplots()`, qui renvoie deux choses :
- la **figure** (`Figure`) : l'image entière ;
- un ou plusieurs **graphiques** (`Axes`), placés dans la figure. `plt.subplots(1, 2)` crée une figure avec 1 ligne et 2
  colonnes de graphiques, et renvoie donc une liste de deux graphiques.

On dessine ensuite sur chaque graphique avec ses méthodes : `.plot()`, `.bar()`... comme avec `plt`. Seule différence :
pour le titre et les axes, les méthodes commencent par `set_` (`.set_title()`, `.set_xlabel()`...).

In [ ]:
figure, graphiques = plt.subplots(1, 2, figsize=(10, 4))  # figsize : la taille de la figure (largeur, hauteur)

graphiques[0].plot(x, y)
graphiques[0].set_title("Une courbe")

graphiques[1].bar(x, z, color="tab:orange")
graphiques[1].set_title("Des barres")
graphiques[1].invert_yaxis()  # on peut aussi inverser un axe !

figure.tight_layout()  # ajuste les espaces pour que rien ne se chevauche
plt.show()

## 3. pandas

Pour des tableaux de données plus conséquents, lire un CSV ligne par ligne et convertir chaque valeur devient vite
fastidieux. Le package [pandas](https://pandas.pydata.org/) rend tout cela beaucoup plus simple. Il est construit autour
d'un objet central, le **DataFrame** : un tableau, avec des lignes et des colonnes nommées, comme dans un tableur.

Par convention, on l'importe sous le nom `pd`. La fonction `pd.read_csv()` lit un CSV et renvoie un DataFrame ; elle
reconnaît automatiquement les nombres. Notre fichier `other_data.csv` n'a pas de ligne d'en-tête : on le précise avec
`header=None`, et on donne nous-mêmes le nom des colonnes avec `names`.

In [ ]:
import pandas as pd

chemin = os.path.join(os.getcwd(), "other_data.csv")

data = pd.read_csv(chemin, header=None, names=["x", "y", "class"])

print(data)

Quelques opérations de base sur un DataFrame :

| Code | Résultat |
|---|---|
| `data.head()` | les 5 premières lignes |
| `len(data)` | le nombre de lignes |
| `data.describe()` | des statistiques sur chaque colonne (moyenne, minimum, maximum...) |
| `data["x"]` | une colonne |
| `data["x"].mean()`, `.min()`, `.max()`, `.sum()` | des statistiques sur une colonne |
| `data[data["x"] > 0]` | uniquement les lignes qui respectent une condition |
| `data.groupby("class").size()` | le nombre de lignes pour chaque valeur de la colonne `class` |

In [ ]:
print(data.head())
print(len(data), "lignes")
print(data.describe())
print("Moyenne de x :", data["x"].mean())
print(len(data[data["x"] > 0]), "points ont un x positif")
print(data.groupby("class").size())

### Un nuage de points coloré par classe

Chaque point de notre fichier appartient à une classe (1, 2 ou 3). Dessinons un nuage de points avec une couleur par
classe.

`data.groupby("class")` regroupe les lignes selon leur classe. Quand on le parcourt avec une boucle `for`, on obtient à
chaque tour deux choses : la valeur de la classe, et le sous-tableau des lignes de cette classe. On dessine donc un nuage
de points par classe, avec sa propre couleur (choisie dans un dictionnaire) et sa propre étiquette pour la légende.

In [ ]:
couleurs = {
  1: "green",
  2: "blue",
  3: "red"
}

plt.figure(figsize=(8, 8))  # crée une nouvelle figure, carrée

for classe, groupe in data.groupby("class"):
  plt.scatter(
    groupe["x"],
    groupe["y"],
    label=f"Classe {classe}",
    color=couleurs.get(classe, "black")  # get() renvoie "black" si la classe n'est pas dans le dictionnaire
  )

plt.legend()
plt.title("Trois spirales")
plt.show()

> 💡 pandas peut aussi dessiner directement : `data.plot.scatter(x="x", y="y")` ou `data["x"].plot.hist()`. Il utilise
> matplotlib en coulisses.

## Pour aller plus loin

- Le deuxième notebook de cette séance, **Atelier Python Session 7 - Data vis 2 - scikit-learn**, montre comment
  visualiser des données qui ont beaucoup plus que deux colonnes.
- Pour vous entraîner (avec vérification automatique et corrigé) : [Atelier Python Session 7 - Exercices.ipynb](https://colab.research.google.com/github/stage-to-data/hackspace-shs/blob/main/cours-python-resources/session-7/Atelier%20Python%20Session%207%20-%20Exercices.ipynb).